In [ ]:
!git clone https://github.com/AshishJangra27/Face-Generator-with-GAN

import tensorflow as tf
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Load generator
generator = tf.keras.models.load_model(
    '/content/Face-Generator-with-GAN/generator_700.h5',
    compile=False
)
print("✅ Generator Loaded")

In [ ]:
!git clone https://github.com/AshishJangra27/Gender-Style-Transfer

import numpy as np

# Load gender direction vector
gender_vector = np.load("/content/Gender-Style-Transfer/gender_vec.npy")
print("✅ Gender vector loaded with shape:", gender_vector.shape)

In [ ]:
import matplotlib.pyplot as plt

def generate_gender_variations(model, gender_vec, noise_dim=100, steps=20):
    # Base latent vector (random)
    base_noise = tf.random.normal([1, noise_dim])

    # Scalars from -5 (male) to +5 (female)
    alphas = np.linspace(-5, 5, steps)

    fig, axes = plt.subplots(1, steps, figsize=(3*steps, 3))

    for i, alpha in enumerate(alphas):
        # Add gender direction
        variation_noise = base_noise + alpha * gender_vec.reshape(1, -1)

        # Generate image
        generated_image = model(variation_noise, training=False)
        generated_image = (generated_image + 1) / 2.0  # Rescale [-1,1] → [0,1]

        # Plot
        axes[i].imshow(generated_image[0])
        axes[i].axis("off")
        axes[i].set_title(f"{alpha:.1f}")

    plt.tight_layout()
    plt.show()

# Generate variations
generate_gender_variations(generator, gender_vector, steps=20)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont
import imageio
from IPython.display import Image as IPyImage, display

# Force CPU (optional -- remove if you want GPU)
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Paths (adjust if your notebook layout differs)
GEN_MODEL_PATH = "/content/Face-Generator-with-GAN/generator_700.h5"
GENDER_VEC_PATH = "/content/Gender-Style-Transfer/gender_vec.npy"

# Load generator
print("Loading generator from:", GEN_MODEL_PATH)
generator = tf.keras.models.load_model(GEN_MODEL_PATH, compile=False)
print("Generator loaded.")

# Load gender vector
print("Loading gender vector from:", GENDER_VEC_PATH)
gender_vec = np.load(GENDER_VEC_PATH)
gender_vec = np.asarray(gender_vec).astype(np.float32)
print("Loaded gender_vec shape:", gender_vec.shape)

# Ensure gender_vec is shaped (noise_dim,)
noise_dim = gender_vec.shape[-1] if gender_vec.ndim > 1 else gender_vec.shape[0]
gender_vec = gender_vec.reshape(1, -1)  # shape (1, noise_dim)
print("Using noise_dim =", noise_dim)

import matplotlib.pyplot as plt

def generate_face_from_z(z_vec):
    """Return uint8 image (H,W,3) in [0..255] from generator for a single z_vec shaped (1, noise_dim)."""
    z_tf = tf.convert_to_tensor(z_vec, dtype=tf.float32)
    with tf.device('/CPU:0'):
        gen = generator(z_tf, training=False)
    # generator likely outputs in [-1,1] range
    gen = (gen + 1.0) / 2.0  # to [0,1]
    gen = tf.clip_by_value(gen, 0.0, 1.0)
    arr = (gen[0].numpy() * 255).astype(np.uint8)
    # If single channel or shape mismatch, convert to 3-channel
    if arr.ndim == 2:
        arr = np.stack([arr]*3, axis=-1)
    if arr.shape[-1] == 1:
        arr = np.concatenate([arr]*3, axis=-1)
    return arr

# Create base noise (reproducible)
rng = np.random.RandomState(42)
base_noise = rng.normal(size=(1, noise_dim)).astype(np.float32)

# Create alphas from male -> female (you can tweak range)
steps = 10
alphas = np.linspace(-5.0, 5.0, steps)  # negative = male direction, positive = female

# Generate images
images = []
for a in alphas:
    z = base_noise + (a * gender_vec)  # shape (1, noise_dim)
    img = generate_face_from_z(z)
    images.append(img)

# Build a horizontal grid image (1 x steps)
pil_imgs = [Image.fromarray(img) for img in images]

thumb_w = 128
thumb_h = int(pil_imgs[0].height * (thumb_w / pil_imgs[0].width))
pil_imgs = [im.resize((thumb_w, thumb_h), Image.LANCZOS) for im in pil_imgs]

# Create single combined grid (one row)
grid_w = thumb_w * steps
grid_h = thumb_h
grid_base = Image.new("RGB", (grid_w, grid_h), (255,255,255))
for i, im in enumerate(pil_imgs):
    grid_base.paste(im, (i * thumb_w, 0))

# Prepare frames for GIF: highlight each column in turn
frames = []
border = 4  # pixels for highlight border
# Try to load a default font for label; PIL's default may be used if not found
try:
    font = ImageFont.truetype("DejaVuSans.ttf", 14)
except Exception:
    font = ImageFont.load_default()

for i in range(steps):
    frame = grid_base.copy()
    draw = ImageDraw.Draw(frame)
    # highlight rectangle around column i
    x0 = i * thumb_w
    y0 = 0
    x1 = x0 + thumb_w
    y1 = grid_h
    # Draw semi-opaque rectangle behind the highlighted thumbnail
    overlay = Image.new("RGBA", (thumb_w, grid_h), (255,255,255,0))
    ov_draw = ImageDraw.Draw(overlay)
    # translucent fill (light)
    ov_draw.rectangle([0,0,thumb_w,grid_h], fill=(255,255,255,60))
    frame.paste(overlay, (x0, y0), overlay)

    # draw border
    draw.rectangle([x0+border//2, y0+border//2, x1-border//2-1, y1-border//2-1], outline=(255,0,0), width=border)

    # add alpha label above the grid for clarity
    label = f"alpha = {alphas[i]:+.2f}"
    # compute text size (new Pillow)
    bbox = draw.textbbox((0, 0), label, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    tx = max(5, x0 + (thumb_w - text_w)//2)
    ty = 3
    # draw text background for readability
    draw.rectangle([tx-3, ty-1, tx + text_w + 3, ty + text_h + 1], fill=(255,255,255,200))
    draw.text((tx, ty), label, fill=(0,0,0), font=font)

    frames.append(frame)

# Save GIF
gif_path = "/content/grid_face_transformation.gif"
# Convert to palette (imageio handles it) and write frames
imageio.mimsave(gif_path, [np.array(f) for f in frames], duration=0.7)  # 0.7s per frame
print("Saved GIF to:", gif_path)

# Display GIF inline in Colab
display(IPyImage(filename=gif_path))


In [ ]:
import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont
import imageio
import numpy as np
from IPython.display import Image as IPyImage, display

CLASSIFIER_MODEL_PATH = "/kaggle/input/gender-classifier-mobilenet/keras/gender-classifier-mobilenet/1/gender_classifier.keras"
gender_classifier = keras.models.load_model(CLASSIFIER_MODEL_PATH)
print("Gender classifier loaded.")

def predict_gender(image):
    """image: Tensor [H,W,3], float in [0,1]"""
    img = tf.image.resize(image, (224,224))
    img = tf.expand_dims(img, axis=0)  # add batch
    pred = gender_classifier.predict(img, verbose=0)[0][0]
    return "Female" if pred > 0.5 else "Male"

labels = []
for alpha, img in zip(alphas, images):
    img_norm = tf.cast(img, tf.float32) / 255.0  # normalize
    label = predict_gender(img_norm)
    labels.append(label)
    print(f"alpha={alpha:+.2f} → {label}")

frames = []
thumb_w = 128
thumb_h = int(images[0].shape[0] * (thumb_w / images[0].shape[1]))

# Try to load font
try:
    font = ImageFont.truetype("DejaVuSans.ttf", 14)
except Exception:
    font = ImageFont.load_default()

for i, (img, label, alpha) in enumerate(zip(images, labels, alphas)):
    # Resize image for consistency
    pil_img = Image.fromarray(img).resize((thumb_w, thumb_h), Image.LANCZOS)

    # Add label on image
    frame = pil_img.copy()
    draw = ImageDraw.Draw(frame)

    # text: label + alpha
    text = f"{label}\nα={alpha:+.2f}"
    text_w, text_h = draw.textbbox((0,0), text, font=font)[2:]
    draw.rectangle([2, 2, text_w+6, text_h+6], fill=(255,255,255,180))
    draw.text((5, 3), text, fill=(0,0,0), font=font)

    frames.append(frame)

# Save GIF
gif_path = "/content/grid_face_transformation_labeled.gif"
imageio.mimsave(gif_path, [np.array(f) for f in frames], duration=0.7)
print("Saved labeled GIF to:", gif_path)

# Show inline
display(IPyImage(filename=gif_path))

In [ ]:
import kagglehub
from tensorflow import keras
import tensorflow as tf

# Download gender classifier model
path = kagglehub.model_download("ashishjangra27/gender-classifier-mobilenet/keras/gender-classifier-mobilenet")
print("Model files are in:", path)

# Correct model path
model_path = f"{path}/gender_classifier.keras"

# Load model
gender_classifier = keras.models.load_model(model_path)
print("Model loaded.")

# Predict gender of generated images
def predict_gender(image):
    img = tf.image.resize(image, (224,224))
    img = tf.expand_dims(img, axis=0)  # batch dimension
    pred = gender_classifier.predict(img)[0][0]  # single output
    return "Female" if pred > 0.5 else "Male"

In [ ]:
for i, img in enumerate(images):
    img_norm = tf.cast(img, tf.float32) / 255.0
    label = predict_gender(img_norm)
    print(f"Image {i+1}: {label}")